# C5.01 Corpus agenti - Tema 2 - student_06

Scop: ne uitam la `data/cleaned/corpus_youtube_sample_annotated.jsonl` (420 de comentarii deja adnotate),
definim tipologia **T6_intelectual_critic** printr-o regula pe axele de adnotare,
verificam distributia si alegem ~50 de exemple pentru vector store.

**Nu sunt necesare apeluri API** - filtram direct pe baza valorilor din corpus-ul adnotat.

## 1. Setup

In [1]:
from pathlib import Path
import os
import pandas as pd

def find_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / ".env").exists():
            return parent
    raise FileNotFoundError("Nu am gasit .env")

PROJECT_ROOT = find_root(Path.cwd())
os.chdir(PROJECT_ROOT)

DATA_PATH = PROJECT_ROOT / "data" / "cleaned" / "corpus_youtube_sample_annotated.jsonl"
print("ROOT:", PROJECT_ROOT)
print("DATA:", DATA_PATH)

ROOT: c:\Users\georg\OneDrive\Dokument\Claude\Projects\Cursul Inginerie Ai\echochamber-project-team3
DATA: c:\Users\georg\OneDrive\Dokument\Claude\Projects\Cursul Inginerie Ai\echochamber-project-team3\data\cleaned\corpus_youtube_sample_annotated.jsonl


In [2]:
DATA_PATH

WindowsPath('c:/Users/georg/OneDrive/Dokument/Claude/Projects/Cursul Inginerie Ai/echochamber-project-team3/data/cleaned/corpus_youtube_sample_annotated.jsonl')

## 2. Incarcam corpusul

In [3]:
df = pd.read_json(DATA_PATH, lines=True)
df.head(2)

,id,source_channel,channel_family,video_title,text,low_information,pre_filtered,target_specific,target_refined,target_l1,...,repr_pluralist_present,repr_pluralist_strength,dem_procedure_rejected,dem_procedure_defended,call_to_action_present,call_to_action_strength,confidence,discourse_type,discourse_subtype,type_confidence
0,yt_Vekhmz5OPCc_UgzpXOYhxlT3Nqo6h7J4AaABAg,NicusorDanRO,mainstream_actor,🟢 LIVE Declarații de presă susținute la Palatu...,jigodia aia de Georgescu nu lua intrebari inca...,False,False,georgescu,georgescu,Sovereigntist,...,False,0,False,False,False,0,0.9,T3_opozitie_suveranista,opozitie_difuza,medium
1,yt_olpOFshMJD0_UgznnPuVIJ6HcddfPbN4AaABAg,georgesimionoficial,sovereigntist,#democratie #georgesimion #unitate #prosperita...,Mă voi realizați că votul s-a încheiat și Nicu...,False,False,nicusor_dan,nicusor_dan,Instituții stat,...,True,1,False,True,False,0,0.9,T5_pro_democratic_european,aparare_institutionala_procedurala,high


## 3. Vedem structura datelor

In [4]:
len(df)

420

In [5]:
print("Coloane:")
df.columns

Coloane:


Index(['id', 'source_channel', 'channel_family', 'video_title', 'text',
       'low_information', 'pre_filtered', 'target_specific', 'target_refined',
       'target_l1', 'target_l2', 'stance_to_target', 'primary_target_hint',
       'target_confidence', 'inst_neg_present', 'inst_neg_strength',
       'inst_pos_present', 'inst_pos_strength',
       'epist_hidden_coordination_present',
       'epist_hidden_coordination_strength',
       'epist_evidence_verification_present',
       'epist_evidence_verification_strength',
       'geo_anti_external_domination_present',
       'geo_anti_external_domination_strength',
       'geo_pro_external_anchoring_present',
       'geo_pro_external_anchoring_strength', 'repr_personalist_present',
       'repr_personalist_strength', 'repr_pluralist_present',
       'repr_pluralist_strength', 'dem_procedure_rejected',
       'dem_procedure_defended', 'call_to_action_present',
       'call_to_action_strength', 'confidence', 'discourse_type',
       'disco

In [6]:
df["discourse_type"].value_counts(dropna=False)

discourse_type
T3_opozitie_suveranista       70
T5_pro_democratic_european    70
T6_afectiv_pozitional         70
T2_grievance_anti_sistem      70
T1_suport_personalist         70
T4_conspiratie_externalism    70
Name: count, dtype: int64

In [7]:
## Exemplu de text pentru fiecare tip de discurs existent in corpus

In [8]:
for bubble in df["discourse_type"].value_counts().index:
    print("\n" + "="*80)
    print(bubble)
    print("="*80)
    sample = df[df["discourse_type"] == bubble]["text"].dropna().head(3)
    for i, text in enumerate(sample, 1):
        print(f"\n{i}. {text[:200]}")


T3_opozitie_suveranista

1. jigodia aia de Georgescu nu lua intrebari inca din campanie, cand NU AVEA PUTEREA ,cum zic americanii writings were on the wall'' scria clar pe perete ce va face avand puterea

2. Lepra aia care ia interval este non stop lângă el cine o fi 🙊🙈🙉🗣🗝

3. Votam masiv Nicusor Dan! Ai scris intr-o postare pe facebook ca ai fost la 7 dezbateri la care nu s-a prezentat contracandidatul. Te rog sa ne spui cum gasim si noi acele dezbateri, nu de alta dar dup

T5_pro_democratic_european

1. Mă voi realizați că votul s-a încheiat și Nicușor Dan este actualul președinte ales de majoritate prin vot fără incidente confirmat și de CCR?. Din partidul POT s-au retras mai toți membrii importanți

2. VICTORIE Ucraina Forte Zelenschi forte UE Euronato 🇺🇦💯🇺🇦🇺🇦🇪🇺🇺🇦🇺🇦🇺🇦🇪🇺🇺🇦🇺🇦🇺🇦🇪🇺

3. Tipic sunt prinși în flagrant amenință cu legea doar ca din păcate pentru ei ce au făcut cei de la recorder este perfect legal. Țineți tot așa

T6_afectiv_pozitional

1. Dictatorul Putin nu vrea pace, v

## 4. Definim T6_intelectual_critic prin regula pe axe

Corpusul are 420 de comentarii adnotate cu valori pe axele de analiza.
Filtram direct pe baza axelor care caracterizeaza bula intelectual-critica - **zero apeluri API**.

**Logica T6:**
- Prezinta dovezi sau perspective pluraliste: `epist_evidence_verification_strength >= 1` SAU `repr_pluralist_strength >= 1`
- Nu e personalist: `repr_personalist_strength == 0`
- Nu e conspirationist: `epist_hidden_coordination_strength == 0`
- Nu e pur mobilizator: `call_to_action_strength <= 1`

In [9]:
def is_t6(row):
    evidence_ok           = row["epist_evidence_verification_strength"] >= 1
    pluralist_ok          = row["repr_pluralist_strength"] >= 1
    not_personalist       = row["repr_personalist_strength"] == 0
    not_conspiratorial    = row["epist_hidden_coordination_strength"] == 0
    not_pure_mobilization = row["call_to_action_strength"] <= 1
    return (
        (evidence_ok or pluralist_ok)
        and not_personalist
        and not_conspiratorial
        and not_pure_mobilization
    )

t6_mask = df.apply(is_t6, axis=1)
print(f"Candidati T6_intelectual_critic: {t6_mask.sum()} din {len(df)}")
print("\nDistributie discourse_type la candidatii T6:")
print(df[t6_mask]["discourse_type"].value_counts())

Candidati T6_intelectual_critic: 58 din 420

Distributie discourse_type la candidatii T6:
discourse_type
T3_opozitie_suveranista       24
T5_pro_democratic_european    19
T4_conspiratie_externalism    13
T1_suport_personalist          1
T2_grievance_anti_sistem       1
Name: count, dtype: int64


In [10]:
df_t6 = df[t6_mask].copy()
df_t6["discourse_type"]    = "T6_intelectual_critic"
df_t6["discourse_subtype"] = "intelectual_critic"
df_t6["type_confidence"]   = "high"

print(f"df_t6: {len(df_t6)} comentarii")
df_t6[["id", "discourse_type", "epist_evidence_verification_strength", "repr_pluralist_strength", "repr_personalist_strength", "text"]].head(3)

df_t6: 58 comentarii


,id,discourse_type,epist_evidence_verification_strength,repr_pluralist_strength,repr_personalist_strength,text
1,yt_olpOFshMJD0_UgznnPuVIJ6HcddfPbN4AaABAg,T6_intelectual_critic,0,1,0,Mă voi realizați că votul s-a încheiat și Nicu...
34,yt_cok3YTTn8sg_Ugw9joa6s_uddLOSdCZ4AaABAg,T6_intelectual_critic,1,0,0,"Catu sa plateasca, daca nu se pruc probe ale u..."
43,yt__xecHPdEhuI_UgxEpwxIKvuelKuQ0r94AaABAg,T6_intelectual_critic,2,0,0,Acum vă eu pe cei care ati votat altceva decat...


## 5. Alege agentul tau si verifica textele

Student_06 lucreaza pe agentul **Intelectual-critic**.

In [11]:
AGENTS = {
    "Personalist-salvator": {"type": "T1_suport_personalist", "slug": "personalist_salvator", "personality": "devotat, admirativ, sigur", "speaks": "laudativ, emotional, increzator", "definition": "vede liderul ca solutie exceptionala"},
    "Anti-sistem": {"type": "T2_grievance_anti_sistem", "slug": "anti_sistem", "personality": "furios, suspicios, dezamagit", "speaks": "acuzator, moralizator, direct", "definition": "vede institutiile si sistemul ca profund compromise"},
    "Anti-suveranist": {"type": "T3_opozitie_suveranista", "slug": "anti_suveranist", "personality": "critic, vigilent, defensiv", "speaks": "contestatar, mai argumentativ", "definition": "respinge liderii si discursul suveranist"},
    "Conspirationist": {"type": "T4_conspiratie_externalism", "slug": "conspirationist", "personality": "alarmist, hiper-suspicios", "speaks": "speculativ, revelator, totalizant", "definition": "explica evenimentele prin forte ascunse si actori externi"},
    "Pro-european": {"type": "T5_pro_democratic_european", "slug": "pro_european", "personality": "normativ, moderat, legalist", "speaks": "sobru, justificativ, procedural", "definition": "apara regulile, institutiile si ancorarea europeana"},
    "Intelectual-critic": {"type": "T6_intelectual_critic", "slug": "intelectual_critic", "personality": "analitic, sceptic, detasat", "speaks": "argumentativ, referential, nuantat", "definition": "evalueaza critic, cere dovezi, refuza personalizarea sau conspiratia"},
}

MY_AGENT = "Intelectual-critic"
meta = AGENTS[MY_AGENT]
my_df = df_t6.drop_duplicates(subset="text").copy()

print("Agent:", MY_AGENT)
print("Tip discurs:", meta["type"])
print("Texte disponibile:", len(my_df))
my_df[["id", "type_confidence", "discourse_subtype", "text"]].head(2)

Agent: Intelectual-critic
Tip discurs: T6_intelectual_critic
Texte disponibile: 58


,id,type_confidence,discourse_subtype,text
1,yt_olpOFshMJD0_UgznnPuVIJ6HcddfPbN4AaABAg,high,intelectual_critic,Mă voi realizați că votul s-a încheiat și Nicu...
34,yt_cok3YTTn8sg_Ugw9joa6s_uddLOSdCZ4AaABAg,high,intelectual_critic,"Catu sa plateasca, daca nu se pruc probe ale u..."


### Cum verifici textele
Citeste textele afisate mai jos. Daca un text este slab, copiaza ID-ul lui in lista `REMOVE_IDS`.

In [12]:
for _, row in my_df.head(70).iterrows():
    print("=" * 80)
    print("ID:", row["id"])
    print("Confidence:", row["type_confidence"])
    print("Subtype:", row["discourse_subtype"])
    print(row["text"][:700])

ID: yt_olpOFshMJD0_UgznnPuVIJ6HcddfPbN4AaABAg
Confidence: high
Subtype: intelectual_critic
Mă voi realizați că votul s-a încheiat și Nicușor Dan este actualul președinte ales de majoritate prin vot fără incidente confirmat și de CCR?. Din partidul POT s-au retras mai toți membrii importanți căutând alte oportunități în partide PSD sau PNL sau USR ceea ce este firesc dacă le merge mintea de ce să nu ocupe un post bun spre beneficiul cetățenilor mai ales dacă au umbrela unor partide puternice?.
ID: yt_cok3YTTn8sg_Ugw9joa6s_uddLOSdCZ4AaABAg
Confidence: high
Subtype: intelectual_critic
Catu sa plateasca, daca nu se pruc probe ale unei decizii la nivel UE. Si nici decixcizia apriorica de reducere a comenzii bazate pe estimarea corecta a numarului imbecililor antivaccinisti romani nu era posibila. Voiculescu nu, ca s-a opus faptic si oficial deciziei de procurare.
ID: yt__xecHPdEhuI_UgxEpwxIKvuelKuQ0r94AaABAg
Confidence: high
Subtype: intelectual_critic
Acum vă eu pe cei care ati votat altce

In [13]:
# elimin textele slabe si pastrez cele mai bune

REMOVE_IDS = [
    # Texte eliminate dupa inspectie manuala
    "yt_dESUmtdVSzo_UgxTk3icWWFR9hcNxsl4AaABAg",  # corectie factica scurta, informal
    "yt_q6wy7cs5RuU_UgxtjWylQ8u3U0mA5u14AaABAg",  # sarcasm pur fara argumentare
    "yt_R-wmsuFxku4_UgxCDktRr9fVoMyi6u94AaABAg",  # mocking, nu analitic
    "yt_iH8jB4NlV9Y_UgzI-eS_CR-n-EkfQRN4AaABAg",  # agresiv, nu critic-detasat
    "yt_uyByTUNzxJI_UgyiIWtfiRkngJmScDB4AaABAg",  # mobilizare, nu analiza
    "yt_TpUDm4ay-B8_UgyaW4uuTLLn4pifcxd4AaABAg",  # whataboutism superficial
    "yt_agapp0LH1dI_UgywoxX6FCkmmKnqZBR4AaABAg",  # populist, sunet de T1
    "yt_yEuctxNb4O0_Ugx1z3x0Sui0lSBOW3h4AaABAg",  # emotional, nu analitic
    "yt_re0gkFt114A_UgwAcMlgCdx0sDUKzGF4AaABAg",  # se termina cu slogan mobilizator
    "yt_coeP4ouSxV8_Ugwkt6kp-nnhYsJ2GQZ4AaABAg",  # prea scurt (78 ch), aforism izolat
]

clean_df = my_df[~my_df["id"].isin(REMOVE_IDS)].copy()

clean_df["agent"]       = MY_AGENT
clean_df["slug"]        = meta["slug"]
clean_df["personality"] = meta["personality"]
clean_df["speaks"]      = meta["speaks"]
clean_df["definition"]  = meta["definition"]

print("Texte finale:", len(clean_df))
clean_df[["id", "agent", "text"]].head()

Texte finale: 48


,id,agent,text
1,yt_olpOFshMJD0_UgznnPuVIJ6HcddfPbN4AaABAg,Intelectual-critic,Mă voi realizați că votul s-a încheiat și Nicu...
34,yt_cok3YTTn8sg_Ugw9joa6s_uddLOSdCZ4AaABAg,Intelectual-critic,"Catu sa plateasca, daca nu se pruc probe ale u..."
43,yt__xecHPdEhuI_UgxEpwxIKvuelKuQ0r94AaABAg,Intelectual-critic,Acum vă eu pe cei care ati votat altceva decat...
68,yt_iH8jB4NlV9Y_UgxUZokcnI7ENgQqHPZ4AaABAg,Intelectual-critic,18:23 poate sa imi explice și mie un Simionist...
72,yt_yEuctxNb4O0_UgxPNjxaWPqCBL5WUAB4AaABAg,Intelectual-critic,Bun! S-a desfășurat aceasta întâlnire. S-a lua...


### Descrierea agentului

**Descriere T6_intelectual_critic:**

- Agentul intelectual-critic evalueaza institutiile si actorii politici prin prisma dovezilor si a argumentelor, nu a loialitatii sau afectului.
- Tonul este detasat, analitic si uneori ironic.
- Recurge frecvent la comparatii, trimiteri la surse sau contextualizari.
- Se diferentiaza de T4 prin absenta conspirationalismului si de T1 prin lipsa personalizarii.
- Un agent AI care simuleaza aceasta bula ar trebui sa formuleze intrebari retorice bazate pe logica.

In [14]:
OUT_DIR = PROJECT_ROOT / "data" / "bubbles"
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUT_DIR / f"{meta['slug']}.jsonl"
clean_df.to_json(out_path, orient="records", lines=True, force_ascii=False)

print("Salvat:", out_path)
print("Texte exportate:", len(clean_df))

Salvat: c:\Users\georg\OneDrive\Dokument\Claude\Projects\Cursul Inginerie Ai\echochamber-project-team3\data\bubbles\intelectual_critic.jsonl
Texte exportate: 48


In [15]:
verify_df = pd.read_json(out_path, lines=True)
print("Inregistrari in fisier:", len(verify_df))
print("Coloane:", len(verify_df.columns))
verify_df[["id", "agent", "slug", "text"]].head(3)

Inregistrari in fisier: 48
Coloane: 43


,id,agent,slug,text
0,yt_olpOFshMJD0_UgznnPuVIJ6HcddfPbN4AaABAg,Intelectual-critic,intelectual_critic,Mă voi realizați că votul s-a încheiat și Nicu...
1,yt_cok3YTTn8sg_Ugw9joa6s_uddLOSdCZ4AaABAg,Intelectual-critic,intelectual_critic,"Catu sa plateasca, daca nu se pruc probe ale u..."
2,yt__xecHPdEhuI_UgxEpwxIKvuelKuQ0r94AaABAg,Intelectual-critic,intelectual_critic,Acum vă eu pe cei care ati votat altceva decat...


## 7. Reflectie finala

Aceasta sectiune este ceruta pentru Tema 2.

**Agent curatat:** Intelectual-critic (T6_intelectual_critic)

**Ce tipuri de texte am eliminat?**  
Am eliminat 10 texte: comentarii prea scurte (sub 80 de caractere), texte sarcastice fara argumentare reala, un comentariu cu ton agresiv, unul mobilizator (slogan la final) si doua cu caracter populist sau emotional care nu exprimau vocea analitica a agentului.

**Au existat exemple ambigue sau gresit clasificate?**  
Da - cateva texte de pe canalul `georgesimionoficial` treceau filtrul T6 tehnic (cereau dovezi), dar aparau logic pozitia suveranista. Le-am pastrat in majoritate pentru ca structura argumentativa era prezenta.

**Corpusul final reprezinta bine vocea agentului?**  
Da. Cele 48 de texte finale sunt analitice, detasate si cer dovezi sau argumente concrete, fara personalizare in jurul unui lider si fara logica conspiratoare.

---

**Caracterul agentului T6_intelectual_critic:**  
Agentul intelectual-critic este vocea care evalueaza discursul politic prin prisma logicii si a dovezilor, nu a loialitatii sau emotiei. Vorbeste calm si uneori ironic, cere probe concrete, citeaza legislatie sau date si refuza sa accepte afirmatii fara suport. Nu este atasat unui lider sau partid - critica deopotriva suveranistii care fac afirmatii nefondate si institutiile care nu livreaza transparenta. Spre deosebire de bula anti-sistem (T2), nu este furios; spre deosebire de pro-european (T5), nu apara procedurile de dragul procedurilor, ci cere substanta. Intr-o dezbatere, pune intrebari incomode: 'Unde sunt dovezile?', 'Ce program concret propui?', 'Cum verificam asta?'